
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>


# 03 - Window Aggregation in Spark Structured Streaming

This notebook demonstrates advanced concepts of Structured Streaming including stateful operations, state management, streaming joins, and window operations.

### Objectives
- Understand stateful vs stateless operations
- Implement windowed operations
- Perform streaming joins
- Work with late arriving data using watermarks

## REQUIRED - SELECT CLASSIC COMPUTE

Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:

1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

1. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

    - In the drop-down, select **More**.

    - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

1. Find the triangle icon to the right of your compute cluster name and click it.

1. Wait a few minutes for the cluster to start.

1. Once the cluster is running, complete the steps above to select your cluster.

## A. Setup and Data Sources

First, let's create two streaming DataFrames that we'll use throughout our demo:
1. An **orders stream** containing customer orders
2. A **status stream** containing order status updates


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Define schema for orders
orders_schema = StructType([
    StructField("customer_id", LongType(), True),
    StructField("notifications", StringType(), True),
    StructField("order_id", LongType(), True),
    StructField("order_timestamp", LongType(), True)
])

# Define schema for status updates
status_schema = StructType([
    StructField("order_id", LongType(), True),
    StructField("order_status", StringType(), True),
    StructField("status_timestamp", LongType(), True)
])

# Create orders streaming DataFrame
orders_stream = spark.readStream \
    .format("json") \
    .schema(orders_schema) \
    .option("maxFilesPerTrigger", 1) \
    .option("path", "/Volumes/dbacademy_retail/v01/retail-pipeline/orders/stream_json") \
    .load()

# Create status streaming DataFrame
status_stream = spark.readStream \
    .format("json") \
    .schema(status_schema) \
    .option("maxFilesPerTrigger", 1) \
    .option("path", "/Volumes/dbacademy_retail/v01/retail-pipeline/status/stream_json") \
    .load()

# Verify both are streaming DataFrames
print(f"orders_stream is streaming: {orders_stream.isStreaming}")
print(f"status_stream is streaming: {status_stream.isStreaming}")

orders_stream is streaming: True
status_stream is streaming: True


## B. Stateless vs Stateful Operations

Let's look at the difference between stateless and stateful operations:

- **Stateless operations**: Process each record independently (e.g., `select`, `filter`)
- **Stateful operations**: Maintain information across batches (e.g., `groupBy`, `join`)

###1. Stateless Operation Example
Let's apply some simple stateless transformations to our streams:


In [0]:
display(orders_stream)

customer_id,notifications,order_id,order_timestamp
23094,Y,75123,1640392092
23457,N,75124,1640392500
23564,Y,75125,1640394862
23392,N,75126,1640396067
23101,Y,75127,1640399066
23466,N,75128,1640404853
23834,Y,75129,1640407272
23852,Y,75130,1640419989
23483,Y,75131,1640422131
23821,N,75132,1640423697


In [0]:
display(status_stream)

order_id,order_status,status_timestamp
75123,placed,1640392092
75124,placed,1640392500
75125,placed,1640394862
75126,placed,1640396067
75127,placed,1640399066
75128,placed,1640404853
75129,placed,1640407272
75130,placed,1640419989
75131,placed,1640422131
75132,placed,1640423697


In [0]:
# Convert timestamps to a more usable format (stateless operation)
orders_transformed = orders_stream \
    .withColumn("order_time", from_unixtime(col("order_timestamp")).cast("timestamp")) \
    .withColumn("notification_enabled", col("notifications") == "Y")

# Display the transformed stream
display(orders_transformed)

customer_id,notifications,order_id,order_timestamp,order_time,notification_enabled
23094,Y,75123,1640392092,2021-12-25T00:28:12Z,true
23457,N,75124,1640392500,2021-12-25T00:35:00Z,false
23564,Y,75125,1640394862,2021-12-25T01:14:22Z,true
23392,N,75126,1640396067,2021-12-25T01:34:27Z,false
23101,Y,75127,1640399066,2021-12-25T02:24:26Z,true
23466,N,75128,1640404853,2021-12-25T04:00:53Z,false
23834,Y,75129,1640407272,2021-12-25T04:41:12Z,true
23852,Y,75130,1640419989,2021-12-25T08:13:09Z,true
23483,Y,75131,1640422131,2021-12-25T08:48:51Z,true
23821,N,75132,1640423697,2021-12-25T09:14:57Z,false


###2. Stateful Operation Example
Now let's perform some stateful operations that maintain state across batches:

In [0]:
# Count orders by status (stateful aggregation)
status_counts = status_stream \
    .groupBy("order_status") \
    .count() \
    .orderBy(col("count").desc())

display(status_counts)

order_status,count
placed,917
preparing,807
on the way,769
delivered,740
return requested,137
return picked up,117
return processed,109
canceled,85
reported shipping error,39
return canceled,22


## C. Window Operations

Window operations allow us to perform aggregations over time windows. We'll demonstrate:
- Tumbling Windows (fixed, non-overlapping)
- Sliding Windows (overlapping windows)

###1. Tumbling Window Example

Let's count orders per 1-minute tumbling window:

In [0]:
# First, make sure to clean up previous streams with same name
for query in spark.streams.active:
    if query.name == "tumbling_window_counts":
        query.stop()

# Prepare data by ensuring we have a proper timestamp column
status_events = status_stream \
    .withColumn("event_time", from_unixtime(col("status_timestamp")).cast("timestamp"))

# Group by status and 1-minute tumbling windows
tumbling_windows = status_events \
    .groupBy(
        window(col("event_time"), "1 minute"),
        col("order_status")
    ) \
    .count()

# Write to memory for visualization
tumbling_window_query = tumbling_windows.writeStream \
    .format("memory") \
    .outputMode("complete") \
    .queryName("tumbling_window_counts") \
    .start()

In [0]:
%sql
-- Query the tumbling window results
SELECT 
  window.start as window_start,
  window.end as window_end,
  order_status,
  count
FROM tumbling_window_counts
ORDER BY window_start, order_status

window_start,window_end,order_status,count
2021-12-25T00:28:00Z,2021-12-25T00:29:00Z,placed,1
2021-12-25T00:35:00Z,2021-12-25T00:36:00Z,placed,1
2021-12-25T01:14:00Z,2021-12-25T01:15:00Z,placed,1
2021-12-25T01:34:00Z,2021-12-25T01:35:00Z,placed,1
2021-12-25T02:24:00Z,2021-12-25T02:25:00Z,placed,1
2021-12-25T04:00:00Z,2021-12-25T04:01:00Z,placed,1
2021-12-25T04:41:00Z,2021-12-25T04:42:00Z,placed,1
2021-12-25T08:13:00Z,2021-12-25T08:14:00Z,placed,1
2021-12-25T08:48:00Z,2021-12-25T08:49:00Z,placed,1
2021-12-25T09:14:00Z,2021-12-25T09:15:00Z,placed,1


###2. Sliding Window Example
Now let's count orders per 2-minute window, sliding every 1 minute:


In [0]:
# Stop any existing queries with the same name
for query in spark.streams.active:
    if query.name == "sliding_window_counts":
        query.stop()

# Group by status and sliding window
sliding_windows = status_events \
    .groupBy(
        window(col("event_time"), "2 minutes", "1 minute"),
        col("order_status")
    ) \
    .count()

# Write to memory for visualization
sliding_window_query = sliding_windows.writeStream \
    .format("memory") \
    .outputMode("complete") \
    .queryName("sliding_window_counts") \
    .start()

In [0]:
%sql
-- Query the sliding window results
SELECT 
  window.start as window_start,
  window.end as window_end,
  order_status,
  count
FROM sliding_window_counts
ORDER BY window_start, order_status

window_start,window_end,order_status,count
2021-12-25T00:27:00Z,2021-12-25T00:29:00Z,placed,1
2021-12-25T00:28:00Z,2021-12-25T00:30:00Z,placed,1
2021-12-25T00:34:00Z,2021-12-25T00:36:00Z,placed,1
2021-12-25T00:35:00Z,2021-12-25T00:37:00Z,placed,1
2021-12-25T01:13:00Z,2021-12-25T01:15:00Z,placed,1
2021-12-25T01:14:00Z,2021-12-25T01:16:00Z,placed,1
2021-12-25T01:33:00Z,2021-12-25T01:35:00Z,placed,1
2021-12-25T01:34:00Z,2021-12-25T01:36:00Z,placed,1
2021-12-25T02:23:00Z,2021-12-25T02:25:00Z,placed,1
2021-12-25T02:24:00Z,2021-12-25T02:26:00Z,placed,1


## D. Streaming Joins

Let's demonstrate joining our streaming order data with status updates.

In [0]:
# Prepare our streaming DataFrames with proper timestamps
orders_with_time = orders_stream \
    .withColumn("order_time", from_unixtime(col("order_timestamp")).cast("timestamp"))

status_with_time = status_stream \
    .withColumn("status_time", from_unixtime(col("status_timestamp")).cast("timestamp"))

###1. Stream-Static Join
First, let's create a static DataFrame for lookup purposes.


In [0]:
# Create a static lookup table for order status descriptions
status_lookup = spark.createDataFrame([
    ("placed", "Order has been placed"),
    ("preparing", "Order is being prepared"),
    ("on the way", "Order is in transit"),
    ("delivered", "Order has been delivered"),
    ("cancelled", "Order has been cancelled")
], ["order_status", "status_description"])

In [0]:
# Join streaming status data with static status descriptions
enriched_status = status_with_time \
    .join(status_lookup, "order_status")

# Display the joined stream
display(enriched_status)

order_status,order_id,status_timestamp,status_time,status_description
placed,75296,1640990674,2021-12-31T22:44:34Z,Order has been placed
placed,75295,1640987527,2021-12-31T21:52:07Z,Order has been placed
placed,75294,1640987218,2021-12-31T21:46:58Z,Order has been placed
placed,75293,1640984029,2021-12-31T20:53:49Z,Order has been placed
placed,75292,1640974381,2021-12-31T18:13:01Z,Order has been placed
placed,75291,1640971896,2021-12-31T17:31:36Z,Order has been placed
placed,75290,1640963057,2021-12-31T15:04:17Z,Order has been placed
placed,75289,1640960321,2021-12-31T14:18:41Z,Order has been placed
placed,75288,1640958185,2021-12-31T13:43:05Z,Order has been placed
placed,75287,1640953917,2021-12-31T12:31:57Z,Order has been placed


###2. Stream-Stream Join
Now let's join our two streaming DataFrames.

In [0]:
# Stop any existing queries
for query in spark.streams.active:
    if query.name == "order_status_join":
        query.stop()

# Join order stream with status stream on order_id
# Note: We need to limit state buildup for production use
order_status_join = orders_with_time \
    .join(
        status_with_time,
        "order_id"
    )

# Write to memory sink
order_status_join_query = order_status_join.writeStream \
    .format("memory") \
    .outputMode("append") \
    .queryName("order_status_join") \
    .start()

In [0]:
%sql
-- Query the joined data
SELECT 
  order_id, 
  customer_id, 
  order_status,
  notifications,
  order_time,
  status_time
FROM order_status_join
LIMIT 20


order_id,customer_id,order_status,notifications,order_time,status_time
75167,23242,placed,N,2021-12-26T16:05:21Z,2021-12-26T16:05:21Z
75190,23836,placed,N,2021-12-27T17:15:22Z,2021-12-27T17:15:22Z
75232,23273,placed,Y,2021-12-29T05:15:06Z,2021-12-29T05:15:06Z
75264,23312,placed,Y,2021-12-30T17:55:12Z,2021-12-30T17:55:12Z
75167,23242,preparing,N,2021-12-26T16:05:21Z,2021-12-28T10:04:15Z
75167,23242,on the way,N,2021-12-26T16:05:21Z,2021-12-30T22:51:44Z
75167,23242,delivered,N,2021-12-26T16:05:21Z,2021-12-30T21:23:59Z
75190,23836,preparing,N,2021-12-27T17:15:22Z,2021-12-30T02:26:10Z
75190,23836,on the way,N,2021-12-27T17:15:22Z,2021-12-29T14:07:04Z
75190,23836,delivered,N,2021-12-27T17:15:22Z,2021-12-28T23:53:36Z


## E. Handling Late Data with Watermarks
Watermarks help us handle late-arriving data by defining how long to wait for late events.


In [0]:
# Stop any existing queries
for query in spark.streams.active:
    if query.name == "windowed_with_watermark":
        query.stop()

# Add watermark to status events
status_with_watermark = status_events \
    .withWatermark("event_time", "10 minutes")

# Windows with watermark
watermarked_windows = status_with_watermark \
    .groupBy(
        window(col("event_time"), "5 minutes"),
        col("order_status")
    ) \
    .count()

# Write to memory
query5 = watermarked_windows.writeStream \
    .format("memory") \
    .outputMode("complete") \
    .queryName("windowed_with_watermark") \
    .start()

In [0]:
%sql
-- Query the windowed data with watermark
SELECT 
  window.start as window_start,
  window.end as window_end,
  order_status,
  count
FROM windowed_with_watermark
ORDER BY window_start, order_status

window_start,window_end,order_status,count
2021-12-25T00:25:00Z,2021-12-25T00:30:00Z,placed,1
2021-12-25T00:35:00Z,2021-12-25T00:40:00Z,placed,1
2021-12-25T01:10:00Z,2021-12-25T01:15:00Z,placed,1
2021-12-25T01:30:00Z,2021-12-25T01:35:00Z,placed,1
2021-12-25T02:20:00Z,2021-12-25T02:25:00Z,placed,1
2021-12-25T04:00:00Z,2021-12-25T04:05:00Z,placed,1
2021-12-25T04:40:00Z,2021-12-25T04:45:00Z,placed,1
2021-12-25T08:10:00Z,2021-12-25T08:15:00Z,placed,1
2021-12-25T08:45:00Z,2021-12-25T08:50:00Z,placed,1
2021-12-25T09:10:00Z,2021-12-25T09:15:00Z,placed,1


Run the cell below to stop the active streaming queries.

In [0]:
for query in spark.streams.active:
    query.stop()


&copy; 2025 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="blank">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy" target="blank">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use" target="blank">Terms of Use</a> | 
<a href="https://help.databricks.com/" target="blank">Support</a>
